# 1 - The supply chain graph — SOLUTIONS

This notebook describes how supply chain graphs can be entered, edited, and used in the latest version of the Brightway LCA framework. It follows our recommended practice for using Brightway. When in doubt on which commands to use, please [check the cheat sheet](https://docs.brightway.dev/en/latest/content/cheatsheet/index.html), and let us know if you think something there is missing.

The most common approach in life cycle assessment, and the approach that Brightway uses, is to model systems as [graphs](https://en.wikipedia.org/wiki/Graph_(discrete_mathematics)). Graphs have nodes and edges, and data attributes can be attached to both nodes and edges. Here is an example of a graph for a simple product system making a bicycle and the associated LCA objects:

<img src="../images/supply-chain-simple.png">

In our graph, edges are [directed](https://en.wikipedia.org/wiki/Graph_(discrete_mathematics)#Directed_graph) - each edge has a _source_ and a _target_. 

We have also added types to both the nodes and edges. These types are metadata, just like things like names, units, and locations. This metadata helps us make sense of the graph in the context of LCA.

Brightway has a suggested set of labels to use for metadata for the different node and edge types - see the [Brightway interface schemas](https://github.com/brightway-lca/bw_interface_schemas/blob/main/bw_interface_schemas/models.py)

This graph can be logically divided between processes - things that do something, action words, or verbs, and products and elementary flows - the physical things being acted upon, or nouns:

<img src="../images/bipartite.png">

Brightway is designed around flexibility, and so this partition is a convention, not a rule. However, for the sake of your and our sanity, we strongly recommend following this pattern :)

It can be helpful to split an LCA graph into a set of [subgraphs](https://en.wikipedia.org/wiki/Glossary_of_graph_theory#subgraph). For example, we might want to separate the work of two analysts, or separate a foreground and a background system. In Brightway, a subgraph is called a `Database`. Let's add a `Database to our graph:

<img src="../images/with-database.png">

A `Database` is just a collection of nodes - it can be large or small, there aren't any general rules. Edges don't belong to a database, as they can cross from one database to another.

In Brightway, we currently have "graphy"-type methods to access nodes and edges, and non-"graphy"-type methods to access database relationships and LCIA, but you should be thinking of them mentally as being part of a large graph. Let's make that graph:

## Using LLMs in this notebook

You are not expected to memorize Python syntax. Use Claude, ChatGPT, or GitHub Copilot freely to help you write code for the exercises. LLMs work well when you:
- Describe what you want in plain English, including library names (`bw2data`, `bw2calc`)
- Paste error messages directly — the LLM can usually diagnose them immediately
- Ask follow-up questions if the first answer doesn't work

Focus on understanding **what** you want to compute. Let the LLM help with the **how**.

In [ ]:
import bw2data as bd

## Switching graphs — `projects`

A Brightway `project` is a separate graph — completely self-contained, and independent of other projects. This independence can lead to data duplication, but helps keep each project safe from the changes in the others.

We start in the `default` project:

In [ ]:
bd.projects.current

We can change to a new project:

In [ ]:
bd.projects.set_current("Another project")
bd.projects.current

We now don't have any databases or any other data in the graph:

In [ ]:
bd.databases

Let's switch back to our default project. It isn't best practice to do data development in the `default` project, so let's rename it:

In [ ]:
bd.projects.rename_project("Bicycle example")

During development, it is very convenient to just delete everything and rerun the notebook to get the right data. Let's make a shortcut to purge the project:

In [ ]:
I_SCREWED_UP = False

if I_SCREWED_UP:
    try:
        bd.projects.delete_project("Bicycle example", delete_dir=True)
    except ValueError:
        pass

### Exercise 1: Explore your projects

List all projects currently registered in Brightway. How many projects exist? Then check which project is currently active. Try creating a third project called `"Sandbox"` and confirm it appears in the list.

<details><summary>Hint</summary>
Use <code>list(bd.projects)</code> to see all projects. Use <code>bd.projects.set_current("Sandbox")</code> to create and switch to a new project. Don't forget to switch back to <code>"Bicycle example"</code> when done.
</details>

In [ ]:
# SOLUTION
print("All projects:", list(bd.projects))
print("Number of projects:", len(list(bd.projects)))
print("Current project:", bd.projects.current)

# Create and switch to a new Sandbox project
bd.projects.set_current("Sandbox")
print("\nAfter creating Sandbox:")
print("All projects:", list(bd.projects))
print("Sandbox appears:", any(p.name == "Sandbox" for p in bd.projects))

# Switch back to Bicycle example
bd.projects.set_current("Bicycle example")
print("\nBack to:", bd.projects.current)

Make sure we are in the right project before continuing:

In [ ]:
bd.projects.set_current("Bicycle example")

## Databases

A `Database` is a named collection of nodes within the current project. Let's create and register one:

In [ ]:
db = bd.Database("🚲")
# Let the metadata system know this database exists. Not necessary if using a `bw2io` importer.
db.register()

Registering a database allows us to find it in our registry of databases (sorry if that was a bit recursive!):

In [ ]:
bd.databases

Creating a `Database` also created its metadata:

In [ ]:
db.metadata

### Exercise 2: Check your databases

Write a one-liner that prints how many databases are registered in the current project. Then check what keys are stored in `db.metadata` — what information does Brightway track automatically?

<details><summary>Hint</summary>
Use <code>len(bd.databases)</code> for the count. <code>db.metadata</code> is a plain dictionary — you can inspect its keys with <code>db.metadata.keys()</code>.
</details>

In [ ]:
# SOLUTION
print("Number of databases:", len(bd.databases))
print("Metadata keys:", list(db.metadata.keys()))
print("Full metadata:", db.metadata)

## Creating nodes

Our first two nodes — the bicycle itself, and the bicycle production activity.

We are using here some [fixed values for type labels](https://github.com/brightway-lca/brightway2-data/blob/main/bw2data/configuration.py). This is preferable to entering the strings ourselves to avoid human error or inconsistency.

In [ ]:
bicycle = db.new_node(
    name="bicycle",
    unit="number",
    type=bd.labels.product_node_default,
)

bike_production = db.new_node(
    name="bike production",
    location="DK",
    type=bd.labels.process_node_default,
)

bicycle.save()
bike_production.save()

*Question* for reflection: Why does the bicycle production have a location and not a unit (and the opposite for the bicycle)?

*Question*: What other attributes like location could we add to bicycle production?

### Exercise 3: Add metadata to `bike_production`

Add a `comment` field to `bike_production` explaining what it does (e.g. `"Assembles a bicycle from carbon fibre parts in Denmark"`). Then save the node and confirm the comment was stored by reading it back.

<details><summary>Hint</summary>
Nodes act like dictionaries: <code>bike_production['comment'] = "your text"</code>. Don't forget to call <code>bike_production.save()</code> — unsaved changes are lost.
</details>

In [ ]:
# SOLUTION
bike_production['comment'] = "Assembles a bicycle from carbon fibre parts in Denmark"
bike_production.save()

# Verify it was saved by reloading from database
reloaded = bd.get_node(database="🚲", name="bike production")
print("Comment stored:", reloaded['comment'])

## Products and their reference processes

A **product node** (like `bicycle`) is a generic object — the bicycle as a concept, something you can buy, use, or model. It carries properties like name, unit, and categories.

But every product is *associated with* a **reference process** — the process that produces it. The process inherits the meaning of the product (what is being made, in what unit, in what context). When you search for a product in Brightway, you typically navigate *via* its reference process.

This relationship is captured by:
- A **production edge** from the process to the product (with `type=bd.labels.production_edge_default` and `functional=True`)
- The `reference product` field on the process (the name of the product it produces)

This matters for calculation: when Brightway builds the technosphere matrix, it maps **products (rows)** to their **producing processes (columns)**. Each product should have exactly one producing process so the matrix is square.

We can check the relationship with `.producers()`:

> **Note:** `.producers()` will be empty until we add production edges below. We will revisit this after creating the edges.

Let's add the rest of the life cycle inventory:

In [ ]:
natural_gas = db.new_node(
    name='natural gas',
    unit='megajoule',
    type=bd.labels.product_node_default,
)
natural_gas_production = db.new_node(
    name='natural gas production',
    location='NO',
    type=bd.labels.process_node_default,
)

natural_gas.save()
natural_gas_production.save()

Brightway doesn't enforce any uniqueness constraints on fields like `name`. The only fields that must be unique are a combination of the `database` and the node `code`. If we can specify the `code` ourselves, we can run the same cell twice safely:

In [ ]:
cf_production = db.new_node(
    code="cf-production",
    name='carbon fibre production',
    location='DE',
    type=bd.labels.process_node_default,
)
cf = db.new_node(
    code="cf",
    name='carbon fibre',
    unit="kilogram",
    type=bd.labels.product_node_default,
)


cf_production.save()
cf.save()

Brightway allows you complete flexibility to store any additional fields that you want on nodes and edges, but our recommendation is to use the following for fields outside the base set of fields given in [bw_interface_schemas](https://github.com/brightway-lca/bw_interface_schemas/blob/main/bw_interface_schemas/models.py):

* `documentation`
    * `dict[str, str]`, e.g. `node['documentation'] = {"treatment_standards_routes": "from processing of high-energy waste"}`
    * For documentation fields outside the general comment
* `tags`
    * `dict[str, JsonValue]`, e.g. `node['tags'] = {"CN 2024": "http://data.europa.eu/xsp/cn2024/681511000080", "start_year": 2024}`
    * For items which come a pre-defined and finite set of possible values
* `attrs`
    * `dict[str, JsonValue]`2024}`
    * For items which whose values are different for each node, and/or are not know in advance of chosen from a given list
* `properties`
    * `dict[str, JsonValue]`, e.g. `node['properties'] = {"carbon_mass_fraction": 0.5}`
    * Quantitative measure of the process or product

Be careful setting `node['tags'] = {<something>}` as this will overwrite any data that was already given as `tags`. If you aren't sure if `tags` exists, you can do `node.setdefault('tags', {})['<something>'] = '<something>'` ([`setdefault` documentation](https://docs.python.org/3/library/stdtypes.html#dict.setdefault)). 

In [ ]:
co2 = db.new_node(
    name="Carbon Dioxide", 
    categories=('air',),
    tags={'CAS Number': '124-38-9'},
    unit='kilogram',
    type=bd.labels.biosphere_node_default,
)

co2.save()

### Exercise 4: Add a tag to the CO2 node

Add a second tag to the `co2` node recording its molecular weight: key `'molecular_weight_g_mol'`, value `44`. Use `setdefault` so you don't accidentally overwrite the existing `'CAS Number'` tag. Save the node and verify both tags are present.

<details><summary>Hint</summary>
<code>co2.setdefault('tags', {})['molecular_weight_g_mol'] = 44</code> then <code>co2.save()</code>. Check with <code>co2['tags']</code>.
</details>

In [ ]:
# SOLUTION
co2.setdefault('tags', {})['molecular_weight_g_mol'] = 44
co2.save()

print("Tags on co2:", co2['tags'])
assert 'CAS Number' in co2['tags'], "CAS Number was overwritten!"
assert co2['tags']['molecular_weight_g_mol'] == 44
print("Both tags present correctly.")

## Adding edges

We also need to create edges between the nodes. We can do this in many ways, here is one - let's add the production of products by processes:

In [ ]:
bike_production.new_edge(
    amount=1,
    input=bicycle,
    type=bd.labels.production_edge_default,
    functional=True,
).save()
cf_production.new_edge(
    amount=1,
    input=cf,
    type=bd.labels.production_edge_default,
    functional=True,
).save()
natural_gas_production.new_edge(
    amount=1,
    input=natural_gas,
    type=bd.labels.production_edge_default,
    functional=True,
).save()    

Now that we have production edges, we can confirm the product-process relationship:

In [ ]:
# Show which process produces the bicycle
print("Producers of bicycle:", list(bicycle.producers()))

# The production edge carries functional=True to mark it as the reference production
for edge in bicycle.producers():
    print("  Edge functional flag:", edge.get('functional'))
    print("  Producing process:", edge['output'])

The use of `input` is a bit weird in the above - this will change as it is incorrect.

Sometimes we can run the same cell multiple times, and create duplicate exchanges. Brightway **will allow** you to create multiple edges between the same source and target.

*Question* for reflection: Why does Brightway allow such seemingly duplicate edges? Does this reflect real-world conditions?

Brightway has a utility function to fix these errors:

In [ ]:
db.delete_duplicate_exchanges()

### Exercise 5: Count technosphere edges

Run the production edge cell above a second time to create duplicate edges, then call `db.delete_duplicate_exchanges()`. Before deleting, write a one-liner using a list comprehension to count how many production edges `bike_production` currently has.

<details><summary>Hint</summary>
<code>len([e for e in bike_production.exchanges() if e['type'] == bd.labels.production_edge_default])</code>. After deduplication, there should be exactly one.
</details>

In [ ]:
# SOLUTION

# First, create duplicates by re-running the production edge creation
bike_production.new_edge(
    amount=1,
    input=bicycle,
    type=bd.labels.production_edge_default,
    functional=True,
).save()

# Count production edges before deduplication
n_before = len([e for e in bike_production.exchanges() if e['type'] == bd.labels.production_edge_default])
print(f"Production edges before deduplication: {n_before}")

# Delete duplicates
db.delete_duplicate_exchanges()

# Count after
n_after = len([e for e in bike_production.exchanges() if e['type'] == bd.labels.production_edge_default])
print(f"Production edges after deduplication: {n_after}")
print(f"Duplicates removed: {n_before - n_after}")

Now let's add the material and energy inputs:

In [ ]:
bike_production.new_edge(
    amount=2.5, 
    type=bd.labels.consumption_edge_default,
    input=cf
).save()

What about some uncertainty? We use [stats_arrays](https://stats-arrays.readthedocs.io/en/latest/) to model probability distribution functions.

In [ ]:
cf_production.new_edge(
    amount=237.3,  # plus 58 kWh of electricity, in ecoinvent 3.8 
    uncertainty_type=5, 
    minimum=200, 
    maximum=300, 
    type=bd.labels.consumption_edge_default,
    input=natural_gas,
).save()

And our emission of carbon dioxide:

In [ ]:
cf_production.new_edge(
    amount=26.6, 
    uncertainty_type=5, 
    minimum=26,
    maximum=27.2, 
    type=bd.labels.biosphere_edge_default,
    input=co2,
).save()

## Exchange iterators

We have shortcuts to traverse the supply chain graph. For inputs, we have `.technosphere()` and `.biosphere()`; for producing edges, we have `.producers()`, and for edges to other nodes which consume the outputs of our node there is `.consumers()`. You can also get all edges with `.edges()`.

These are all [iterators](https://jakevdp.github.io/WhirlwindTourOfPython/10-iterators.html).

In [ ]:
list(cf_production.technosphere())

In [ ]:
list(cf_production.biosphere())

In [ ]:
list(cf.consumers())

In [ ]:
list(cf_production.producers())

### Exercise 6: Technosphere edge count for `bike_production`

Using a list comprehension, count how many **technosphere** (consumption) edges `bike_production` has. Then print the name and amount of each input.

<details><summary>Hint</summary>
<code>list(bike_production.technosphere())</code> returns all consumption edges. Each edge <code>e</code> has <code>e['amount']</code> and <code>e.input['name']</code>.
</details>

In [ ]:
# SOLUTION
tech_edges = list(bike_production.technosphere())
print(f"Number of technosphere inputs: {len(tech_edges)}")
print()
for edge in tech_edges:
    print(f"  Input: {edge.input['name']}, amount: {edge['amount']}")

## The linearization problem — a preview

In the graph model, a product can *in principle* have multiple producers. Let's demonstrate this by creating a second process that also produces the `bicycle` product:

In [ ]:
# Create a second (hypothetical) producer of the bicycle
bike_production_2 = db.new_node(
    code="bike-production-2",
    name="bike production (aluminium)",
    location="CN",
    type=bd.labels.process_node_default,
)
bike_production_2.save()

bike_production_2.new_edge(
    amount=1,
    input=bicycle,
    type=bd.labels.production_edge_default,
    functional=True,
).save()

print("Producers of bicycle:", [list(e['output']) for e in bicycle.producers()])

Now `bicycle` has **two producers**. This is valid in a graph — real supply chains do have competing suppliers. But it creates a problem:

> **When Brightway builds the technosphere matrix, which producer does it choose?**

The technosphere matrix must be square — one column per product (process), one row per product. So Brightway must pick exactly **one** producing process per product. This is called the **linearization constraint**:

- The rich, many-to-many graph must be flattened so that each product has exactly one producing process.
- In practice, Brightway looks for the production edge marked `functional=True`. If there is more than one such edge pointing to the same product, matrix construction becomes ambiguous.
- Well-formed Brightway databases have exactly one functional producer per product.

Let's clean up by deleting the second producer:

In [ ]:
bike_production_2.delete()

# Confirm only one producer remains
print("Producers of bicycle after cleanup:", list(bicycle.producers()))

We will return to this constraint in **notebook 3 (Building and using matrices in bw2calc)** when we see exactly how the technosphere matrix is assembled, and in notebook 5 (Dynamic Graphs) where we explore ways to relax it.

### Exercise 7: Find multi-producer products in USEEIO

After we load the USEEIO database (next section), come back and write code to find any product node in USEEIO that has more than one outgoing production edge. How many such products exist?

<details><summary>Hint</summary>
Switch to the <code>"USEEIO-1.1"</code> project. Iterate over nodes with <code>type == 'product'</code>. For each, call <code>list(node.producers())</code> and check if <code>len(...) > 1</code>.
</details>

In [ ]:
# SOLUTION — run this cell after the USEEIO section below has been executed
bd.projects.set_current("USEEIO-1.1")
useeio_db = bd.Database("USEEIO-1.1")

multi_producer_products = [
    node for node in useeio_db
    if node['type'] == 'product' and len(list(node.producers())) > 1
]

print(f"Products with more than one producer: {len(multi_producer_products)}")
for p in multi_producer_products:
    print(f"  {p['name']} — {len(list(p.producers()))} producers")

# Switch back to bicycle example for the rest of the exercises
bd.projects.set_current("Bicycle example")

## A quick LCA calculation preview

This is a life cycle inventory, and is enough to do an LCI calculation. Let's do a quick calculation:

In [ ]:
import bw2calc as bc

Don't worry about the syntax here right now, we will talk about it later.

In [ ]:
bd.projects.set_current("Bicycle example")
functional_unit, data_objs, _ = bd.prepare_lca_inputs({bicycle: 1}, remapping=False)

In [ ]:
lca = bc.LCA(demand=functional_unit, data_objs=data_objs)
lca.lci()

*Question*: How much CO2 should be emitted by our functional unit? You can do this calculation by examining the graph manually.

In [ ]:
lca.inventory[lca.dicts.biosphere[co2.id], :].sum()

### Exercise 8: Recalculate with a different carbon fibre amount

Find the consumption edge between `bike_production` and `cf`, change the amount to `3` kg (instead of 2.5 kg), save the edge, and rerun the LCA. What is the new total CO2 emission? (Tip: you will need to rebuild the LCA object from scratch to pick up the change.)

<details><summary>Hint</summary>
Use <code>list(bike_production.technosphere())</code> to find the edge. Set <code>edge['amount'] = 3</code> and <code>edge.save()</code>. Then re-run from <code>bd.prepare_lca_inputs</code> onward.
</details>

In [ ]:
# SOLUTION

# Find and modify the carbon fibre consumption edge
for edge in bike_production.technosphere():
    if edge.input['name'] == 'carbon fibre':
        old_amount = edge['amount']
        edge['amount'] = 3
        edge.save()
        print(f"Changed carbon fibre amount from {old_amount} to {edge['amount']} kg")
        break

# Rebuild LCA to pick up the change
functional_unit, data_objs, _ = bd.prepare_lca_inputs({bicycle: 1}, remapping=False)
lca = bc.LCA(demand=functional_unit, data_objs=data_objs)
lca.lci()

new_co2 = lca.inventory[lca.dicts.biosphere[co2.id], :].sum()
print(f"New CO2 emission: {new_co2:.4f} kg")
# Expected: 3 kg CF * 26.6 kg CO2/kg CF = 79.8 kg CO2
print(f"Expected: {3 * 26.6:.1f} kg CO2")

# Restore original amount
for edge in bike_production.technosphere():
    if edge.input['name'] == 'carbon fibre':
        edge['amount'] = 2.5
        edge.save()
print("Restored carbon fibre amount to 2.5 kg")

Finally, if we want to do impact assessment, we need some nodes to represent impact categories, and edges to represent characterization factors:

<img src="../images/with-lcia.png">

## LCIA

To define characterization nodes and edges, we use a different data structure:

In [ ]:
import stats_arrays as sa

In [ ]:
ipcc = bd.Method(('IPCC',))
ipcc.write([
    (co2, {'amount': 1, 'uncertainty_type': sa.NormalUncertainty.id, 'loc': 1, 'scale': 0.05}),
])

We can now do a full LCIA, not just an LCI:

In [ ]:
functional_unit, data_objs, _ = bd.prepare_lca_inputs({bicycle: 1}, method=('IPCC',), remapping=False)

In [ ]:
lca = bc.LCA(demand=functional_unit, data_objs=data_objs)
lca.lci()
lca.lcia()
lca.score

To use uncertainty, we tell the `LCA` object to use the probability distributions:

In [ ]:
import pandas as pd

In [ ]:
lca = bc.LCA(demand=functional_unit, data_objs=data_objs, use_distributions=True)
lca.lci()
lca.lcia()

df = pd.DataFrame([{'score': lca.score} for _ in zip(lca, range(25))])
df.hist()

### Exercise 9: Create a new impact category

Create a new LCIA method called `('Resource Depletion', 'Natural Gas')` with a characterization factor of `1` for `natural_gas` (no uncertainty). Calculate the LCIA score for producing 1 bicycle. What value do you expect, and does the result match?

<details><summary>Hint</summary>
Use <code>bd.Method(('Resource Depletion', 'Natural Gas'))</code> and <code>.write([(natural_gas, {'amount': 1})])</code>. Then prepare LCA inputs with <code>method=('Resource Depletion', 'Natural Gas')</code> and run <code>lca.lci(); lca.lcia(); lca.score</code>.
</details>

In [ ]:
# SOLUTION

# Create the new impact method
resource_depletion = bd.Method(('Resource Depletion', 'Natural Gas'))
resource_depletion.write([
    (natural_gas, {'amount': 1}),
])

# Prepare inputs with the new method
functional_unit, data_objs, _ = bd.prepare_lca_inputs(
    {bicycle: 1},
    method=('Resource Depletion', 'Natural Gas'),
    remapping=False
)

lca = bc.LCA(demand=functional_unit, data_objs=data_objs)
lca.lci()
lca.lcia()

print(f"LCIA score: {lca.score:.2f} MJ of natural gas")
# Expected: 1 bicycle needs 2.5 kg carbon fibre,
# which needs 237.3 MJ natural gas. So score = 237.3 MJ.
print(f"Expected: {2.5 * 237.3:.1f} MJ")

## Exercise: Build a steel bicycle

Create a new bicycle made of steel. You will need some coal and some iron ore mining for the steel, and some steel for the bicycle.

Your use of steel consumed some iron ore, a natural resource. In a **new database**, create a biosphere flow for this iron ore, and add the iron ore flow. You might not be sure about the numbers - you can reflect that in the uncertainty you assign to the exchanges.

In [ ]:
# SOLUTION

# Create a biosphere database for the iron ore elementary flow
bio_db = bd.Database("biosphere")
bio_db.register()

iron_ore_bio = bio_db.new_node(
    name="Iron ore, in ground",
    categories=('natural resource', 'in ground'),
    unit='kilogram',
    type=bd.labels.biosphere_node_default,
)
iron_ore_bio.save()

# Create the foreground technosphere nodes
steel_bike = db.new_node(
    name="steel bicycle",
    unit="number",
    type=bd.labels.product_node_default,
)
steel_bike_production = db.new_node(
    name="steel bike production",
    location="DE",
    type=bd.labels.process_node_default,
)
steel = db.new_node(
    name="steel",
    unit="kilogram",
    type=bd.labels.product_node_default,
)
steel_production = db.new_node(
    name="steel production",
    location="DE",
    type=bd.labels.process_node_default,
)
coal = db.new_node(
    name="coal",
    unit="kilogram",
    type=bd.labels.product_node_default,
)
coal_mining = db.new_node(
    name="coal mining",
    location="US",
    type=bd.labels.process_node_default,
)

for node in [steel_bike, steel_bike_production, steel, steel_production, coal, coal_mining]:
    node.save()

# Production edges
steel_bike_production.new_edge(amount=1, input=steel_bike,
    type=bd.labels.production_edge_default, functional=True).save()
steel_production.new_edge(amount=1, input=steel,
    type=bd.labels.production_edge_default, functional=True).save()
coal_mining.new_edge(amount=1, input=coal,
    type=bd.labels.production_edge_default, functional=True).save()

# Technosphere consumption edges
# A steel bicycle needs ~15 kg steel
steel_bike_production.new_edge(
    amount=15, uncertainty_type=5, minimum=12, maximum=18,
    type=bd.labels.consumption_edge_default, input=steel).save()

# Producing 1 kg steel needs ~0.5 kg coal (rough estimate)
steel_production.new_edge(
    amount=0.5, uncertainty_type=5, minimum=0.4, maximum=0.65,
    type=bd.labels.consumption_edge_default, input=coal).save()

# Iron ore biosphere flow from steel production (~1.5 kg ore per kg steel)
steel_production.new_edge(
    amount=1.5, uncertainty_type=5, minimum=1.2, maximum=1.8,
    type=bd.labels.biosphere_edge_default, input=iron_ore_bio).save()

print("Steel bicycle system created.")
print("Technosphere inputs to steel_bike_production:")
for e in steel_bike_production.technosphere():
    print(f"  {e.input['name']}: {e['amount']} {e.input.get('unit', '')}")
print("Biosphere flows from steel_production:")
for e in steel_production.biosphere():
    print(f"  {e.input['name']}: {e['amount']} {e.input.get('unit', '')}")

## Exercise: LCIA for iron ore

Create a new LCIA method for your iron ore consumption. Calculate the LCIA result you should get, and then verify that you have the correct value.

In [ ]:
# SOLUTION

iron_ore_method = bd.Method(('Resource Depletion', 'Iron Ore'))
iron_ore_method.write([
    (iron_ore_bio, {'amount': 1}),
])

functional_unit, data_objs, _ = bd.prepare_lca_inputs(
    {steel_bike: 1},
    method=('Resource Depletion', 'Iron Ore'),
    remapping=False
)

lca = bc.LCA(demand=functional_unit, data_objs=data_objs)
lca.lci()
lca.lcia()

print(f"LCIA score: {lca.score:.2f} kg iron ore")
# Expected: 15 kg steel * 1.5 kg ore/kg steel = 22.5 kg iron ore
expected = 15 * 1.5
print(f"Expected (manual): {expected:.1f} kg iron ore")

## Searching through the database

In addition to storing and using nodes and edges, our graph database can be searched in different ways. Let's show this with a larger database.

We can use a shortcut to install some data:

In [ ]:
import bw2io as bi
bi.install_project("USEEIO-1.1", overwrite_existing=True)

# If that doesn't work for whatever reason, we can import the original data with this:
# bi.useeio11()

In [ ]:
bd.projects.set_current("USEEIO-1.1")

In [ ]:
bd.databases

In [ ]:
db = bd.Database("USEEIO-1.1")
db.name

We can search with the 'search' function.

In [ ]:
fun = db.search("amusement")[0]
fun['name'] = 'fun'
fun.save()

In [ ]:
db.search('amusement')

In [ ]:
{node['name'] for node in db if node['type'] == 'product'}

### Exercise 10: Search for motor products

Find all **product** nodes in USEEIO whose name contains the word `'motor'` (case-insensitive). How many are there? Print their names.

<details><summary>Hint</summary>
Use a set or list comprehension: <code>[node['name'] for node in db if node['type'] == 'product' and 'motor' in node['name'].lower()]</code>.
</details>

In [ ]:
# SOLUTION
motor_products = [
    node['name']
    for node in db
    if node['type'] == 'product' and 'motor' in node['name'].lower()
]

print(f"Product nodes containing 'motor': {len(motor_products)}")
for name in sorted(motor_products):
    print(f"  {name}")

## Interacting with the graph

In [ ]:
moo = bd.get_node(name='Cattle ranches and feedlots', type='product')

In [ ]:
type(moo) == bd.Node

We can assign any attributes to nodes (and to edges)

In [ ]:
moo['moo'] = 'loud'

Note that this attributes are not saved to the database by default - we have to tell Brightway to save changed data!

We have some attributes which are common to all inventory databases

In [ ]:
moo['categories'], moo['location'], moo['unit']

The node classes act like dictionaries, and raise error for missing keys

In [ ]:
moo['missing']

### Exercise 11: Explore cattle ranches technosphere inputs

Get the **process** node (not the product node) for `'Cattle ranches and feedlots'` and print the names and amounts of all its technosphere inputs. Which input has the highest amount?

<details><summary>Hint</summary>
Use <code>bd.get_node(name='Cattle ranches and feedlots', type='process')</code>. Then iterate over <code>node.technosphere()</code> and access <code>e.input['name']</code> and <code>e['amount']</code>.
</details>

In [ ]:
# SOLUTION
cattle_process = bd.get_node(name='Cattle ranches and feedlots', type='process')

inputs = [(e.input['name'], e['amount']) for e in cattle_process.technosphere()]
inputs_sorted = sorted(inputs, key=lambda x: x[1], reverse=True)

print("Technosphere inputs (sorted by amount, descending):")
for name, amount in inputs_sorted:
    print(f"  {name}: {amount:.4f}")

print(f"\nHighest input: {inputs_sorted[0][0]} ({inputs_sorted[0][1]:.4f})")

## Exercise: Tag long-named product nodes

Iterate through all `product` nodes in the US EEIO and tag every node whose combined name and unit is more than 40 characters long with `"long" = True`.

In [ ]:
# SOLUTION
count = 0
for node in db:
    if node['type'] == 'product':
        combined = node.get('name', '') + node.get('unit', '')
        if len(combined) > 40:
            node.setdefault('tags', {})['long'] = True
            node.save()
            count += 1

print(f"Tagged {count} product nodes as 'long'")

# Verify a few
long_nodes = [node for node in db if node['type'] == 'product' and node.get('tags', {}).get('long')]
print(f"Verification: found {len(long_nodes)} tagged nodes")
for node in long_nodes[:3]:
    print(f"  '{node['name']}' + '{node.get('unit','')}' = {len(node.get('name','') + node.get('unit',''))} chars")

## Exchange iterators in USEEIO

The US EEIO is normalized to the production of one USD. It can be interesting to sum the costs of the inputs:

In [ ]:
bd.projects.set_current("USEEIO-1.1")

In [ ]:
sum([o['amount'] for o in bd.get_node(name='Cattle ranches and feedlots', type='process').technosphere()])

# Contribution analysis

Let's show a little bit of what Brightway can do. We can compare the correlation of LCA scores across a variety of categories.

There is an automatic way to do this in Brightway, but we can also program it manually to see how it works.

Stop for a bit and think about what one would need to calculate LCA scores for 380 products and ~10 impact categories.

In [ ]:
products_in_order = [obj for obj in db if obj['type'] == 'product']
categories_in_order = [method for method in bd.methods if method[0] == 'Impact Potential']

In general, in Brightway there is *one secret* to getting good performance: Don't rebuild matrices unless you really need to. Rebuilding a matrix is not slow, but the time adds up if you do it a lot. But most importantly, if you are using `pypardiso` (normally everything except for ARM machines), and you keep the technosphere matrix the same, it will secretly remember all the preparation work it did to solve the linear system and you will get a factor of 100 speed increase on subsequent calculations.

So, in this case we will use one `LCA` object, and use the `lci` method repeatedly.

In [ ]:
import numpy as np

results = np.zeros((len(products_in_order), len(categories_in_order)))

def get_lcia_scores(products, categories, results):
    lca = bc.LCA({products[0]: 1}, categories[0])
    lca.lci()
    lca.lcia()
    
    method_matrices = [lca.characterization_matrix.copy()]
    
    for other_method in categories[1:]:
        # Only build each characterization matrix once instead of once per product
        lca.switch_method(other_method)
        method_matrices.append(lca.characterization_matrix.copy())
    
    for i, product in enumerate(products):
        lca.lci({product.id: 1})
        for j, characterization_matrix in enumerate(method_matrices):
            results[i, j] = (characterization_matrix * lca.inventory).sum()
    
    return results

In [ ]:
from time import time

start = time()
results = get_lcia_scores(products_in_order, categories_in_order, results)
print(time() - start)

## Exercise: Benchmark matrix reuse

Compare how long it would take to calculate LCA results for 3 products and 3 impact categories if you had to create a new LCA object each time (instead of reusing the same one).

In [ ]:
# SOLUTION

sample_products = products_in_order[:3]
sample_categories = categories_in_order[:3]

# --- Slow approach: new LCA object each time ---
start_slow = time()
for product in sample_products:
    for category in sample_categories:
        lca_slow = bc.LCA({product: 1}, category)
        lca_slow.lci()
        lca_slow.lcia()
        _ = lca_slow.score
slow_time = time() - start_slow

# --- Fast approach: reuse LCA object ---
start_fast = time()
lca_fast = bc.LCA({sample_products[0]: 1}, sample_categories[0])
lca_fast.lci()
lca_fast.lcia()
for product in sample_products:
    lca_fast.lci({product.id: 1})
    for category in sample_categories:
        lca_fast.switch_method(category)
        lca_fast.lcia()
        _ = lca_fast.score
fast_time = time() - start_fast

print(f"Slow (new LCA each time): {slow_time:.3f} s")
print(f"Fast (reuse LCA object):  {fast_time:.3f} s")
print(f"Speedup factor: {slow_time / fast_time:.1f}x")

In [ ]:
from scipy import stats

def create_correlation_matrix(scores_array):
    num_methods = scores_array.shape[1]
    correlations = np.zeros((num_methods, num_methods))

    for row in range(num_methods):
        for col in range(num_methods):
            if col <= row:
                continue                               # Only need to compute correlation once
            dataset_1 = scores_array[:, row]
            dataset_2 = scores_array[:, col]
            mask = (dataset_1 != 0) * (dataset_2 != 0) # Ignore activities that have zero score
            corr = stats.kendalltau( # Get tau value, drop p-statistic
                dataset_1[mask], 
                dataset_2[mask]
            )[0]
            if np.isnan(corr):
                correlations[row, col] = 0
            else:
                correlations[row, col] = corr

    correlations = correlations + correlations.T       # Make sorting easier by adding filling in lower left triangle
    return correlations

In [ ]:
correlation_matrix = create_correlation_matrix(results)

In [ ]:
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt

fig = plt.gcf()
fig.set_size_inches(12, 12)

masked_correlation = np.ma.array(correlation_matrix, mask=correlation_matrix == 0).T
plt.pcolor(masked_correlation, cmap=plt.cm.cubehelix_r)
plt.colorbar()
plt.ylim(None, correlation_matrix.shape[1])
plt.xlim(None, correlation_matrix.shape[0])
plt.xticks(np.arange(0.5, 10), [obj[1] for obj in categories_in_order])
plt.yticks(np.arange(0.5, 10), [obj[1] for obj in categories_in_order])
plt.tight_layout()

In [ ]:
for category in categories_in_order:
    print(category[1], bd.methods[category]['description'])